In [ ]:
#admin_modules.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    </head>

    <body>
        <header class="header-bar">
            <div class="header-left">
                <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Zeitbuchungssystem</a>
            </div>
        </header>

        <h2>Module festlegen</h2>

        <form method="post" action="{% url 'admin_modules' %}?user_id={{ user_id }}">
            {% csrf_token %}

            <input type="hidden" name="user_id" value="{{ user_id }}">

            <label for="module">Module (je Zeile eines):</label><br>
            <textarea id="module" name="module" rows="8" cols="40">{{ modules_text }}</textarea>

            <br><br>
            <button type="submit">Speichern</button>
        </form>
    </body>
</html>

In [ ]:
#admin_request_list.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    </head>

    <body>
        <header class="header-bar">
            <div class="header-left">
                <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Zeitbuchungssystem</a>
            </div>
        </header>

        <h2>Offene Anträge</h2>

        <table>
            <tr>
                <th>E-Mail</th>
                <th>Art des Antrags</th>
                <th>Aktion</th>
            </tr>

            {% for u in requests %}
            <tr>
                <td>{{ u.email }}</td>

                <td>
                    {% if u.vip_request %} VIP-Antrag {% endif %}
                    {% if u.admin_request %} Admin-Antrag {% endif %}
                </td>

                <td>
                    {% if u.vip_request %}
                        <a href="{% url 'genehmige_vip' u.email %}?user_id={{ user_id }}">VIP genehmigen</a>
                    {% endif %}

                    {% if u.admin_request %}
                        <a href="{% url 'genehmige_admin' u.email %}?user_id={{ user_id }}">Admin genehmigen</a>
                    {% endif %}
                </td>
            </tr>

            {% empty %}
            <tr>
                <td colspan="3">Keine offenen Anträge.</td>
            </tr>
            {% endfor %}
        </table>
    </body>
</html>

In [ ]:
#admin_user_list.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    </head>

    <body>
        <header class="header-bar">
            <div class="header-left">
                <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Zeitbuchungssystem</a>
            </div>
        </header>

        <h2>Alle Benutzer</h2>

        <table>
            <tr>
                <th>Username</th>
                <th>Email</th>
                <th>Rolle</th>
                <th>Status</th>
                <th>Aktion</th>
            </tr>

            {% for u in users %}
            <tr>
                <td>{{ u.username }}</td>
                <td>{{ u.email }}</td>
                <td>{{ u.role }}</td>

                <td>
                    {% if u.is_active %}
                        Aktiv
                    {% else %}
                        Gesperrt
                    {% endif %}
                </td>

                <td>
                    {% if u.is_active %}
                        <a href="{% url 'user_sperren' u.id %}?user_id={{ user_id }}">Sperren</a>
                    {% else %}
                        <a href="{% url 'user_entsperren' u.id %}?user_id={{ user_id }}">Entsperren</a>
                    {% endif %}
                </td>
            </tr>
            {% endfor %}
        </table>
    </body>
</html>

In [ ]:
#arbeitsberichte.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
        <title>Zeitbuchungssystem</title>
    </head>

    <body>
        <header class="header-bar">
            <div class="header-left">
                <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Zeitbuchungssystem</a>
            </div>

            <div class="header-right">

            {% if role == "admin" %}
                <strong>Admin-Bereich</strong>
                <a href="{% url 'admin_request_list' %}?user_id={{ user_id }}">Anträge ansehen</a>
                <a href="{% url 'admin_user_list' %}?user_id={{ user_id }}">Benutzer verwalten</a>
                <a href="{% url 'admin_modules' %}?user_id={{ user_id }}">Module verwalten</a>
            {% endif %}

            {% if user %}
                <a href="{% url 'gesamtuebersicht' %}?user_id={{ user_id }}">Gesamtübersicht</a>
                
                {% if role == "einfach" %}
                    <a href="{% url 'bestaetige_vip' %}?user_id={{ user_id }}">VIP beantragen</a>
                {% endif %}

                {% if role == "vip" %}
                    <a href="{% url 'bestaetige_admin' %}?user_id={{ user_id }}">Admin beantragen</a>
                {% endif %}

                <a href="{% url 'logout' %}?user_id={{ user_id }}" class="logout-button">Logout</a>
            {% endif %}
            </div>
        </header>

        <div style="height: 15px; background-color: lightgray;"></div>


        <h4>Meine Arbeitsberichte</h4>

        <form method="post" action="?user_id={{ user_id }}">
            {% csrf_token %}
            <input type="hidden" name="user_id" value="{{ user_id }}">

            <label for="modul">Modul:</label>
            <select name="modul">
                {% for m in modules %}
                    <option value="{{ m }}">{{ m }}</option>
                {% endfor %}
            </select>

            <label for="datum">Datum:</label>
            <input type="text" id="datum" name="datum" required>

            <label for="minuten">Arbeitszeit (min):</label>
            <input type="number" id="minuten" name="minuten" required>

            <label for="inhalt">Bericht:</label>
            <textarea id="inhalt" name="inhalt" required></textarea>

            <button type="submit">Absenden</button>
        </form>

        {% if role == "vip" or role == "admin" %}
            <a href="{% url 'download_json' %}?user_id={{ user_id }}">JSON herunterladen</a>
            <a href="{% url 'download_csv' %}?user_id={{ user_id }}">CSV herunterladen</a>
            <a href="{% url 'download_xml' %}?user_id={{ user_id }}">XML herunterladen</a>
            
            <form action="{% url 'upload_data' %}?user_id={{ user_id }}" method="post" enctype="multipart/form-data">
                {% csrf_token %}
                <input type="hidden" name="user_id" value="{{ user_id }}">
                <input type="file" name="datei" required>
                <button type="submit">Daten hochladen</button>
            </form>
        {% endif %}

        <table class="berichte">
            <tr>
                <th>Modul</th>
                <th>Datum</th>
                <th>Arbeitszeit (min)</th>
                <th>Bericht</th>
                <th>Aktion</th>
            </tr>
            
        {% for bericht in arbeitsberichte %}
        <tr>
            <td>{{ bericht.modul }}</td>
            <td>{{ bericht.datum }}</td>
            <td style="text-align: right;">{{ bericht.minuten }}</td>
            <td>{{ bericht.inhalt }}</td>

            <td>
                <a href="{% url 'bericht_loeschen' forloop.counter0%}?user_id={{ user_id }}">
                    Löschen
                </a>
            </td>
        </tr>

        {% empty %}
        <tr>
            <td colspan="5">Noch keine Einträge.</td>
        </tr>
        {% endfor %}
        </table>

    </body>
</html>

In [ ]:
#bestaetige_admin.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    </head>

    <body>
        <h2>Admin beantragen</h2>
        <p>Möchtest du wirklich Admin werden?</p>

        <form action="{% url 'request_admin' %}?user_id={{ user_id }}" method="post">
            {% csrf_token %}
            <input type="hidden" name="user_id" value="{{ user_id}}">
            <button type="submit">Ja, beantragen</button>
        </form>

        <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Nein, zurück</a>
    </body>
</html>

In [ ]:
#bestaetige_vip.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    </head>

    <body>
        <h2>VIP beantragen</h2>
        <p>Möchtest du wirklich VIP werden?</p>

        <form action="{% url 'request_vip' %}?user_id={{ user_id }}" method="post">
            {% csrf_token %}
            <input type="hidden" name="user_id" value="{{ user_id }}">
            <button type="submit">Ja, beantragen</button>
        </form>

        <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Nein, zurück</a>
    </body>
</html>

In [ ]:
#gesamtuebersicht.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
        <title>Modul-Zeitübersicht</title>
    </head>

    <body>
        <header class="header-bar">
            <div class="header-left">
                <a href="{% url 'arbeitsberichte' %}?user_id={{ user_id }}">Zeitbuchungssystem</a>
            </div>
        </header>
        
        <div style="height: 15px; background-color: lightgray;"></div>
    
    <div class = "bericht-A4">
        <h2>Modul-Zeitübersicht</h2>
        <p><button class="print-button" onclick="window.print()">Drucken</button></p>

        <center>
        <table class="modul-zeitübersicht" style="width: 85%;">
            <tr>
                <th>Modul</th>
                <th>Zeit</th>
                <th>von gesamter Arbeitszeit in %</th>
            </tr>

            {% for eintrag in daten %}
            <tr>
                <td>{{ eintrag.modul }}</td>
                <td>{{ eintrag.minuten }}</td>
                <td>{{ eintrag.prozent }}</td>
            </tr>
            {% empty %}
            <tr>
                <td colspan="3">Noch keine Einträge.</td>
            </tr>
            {% endfor %}
        </table>
        </center>
    </div>
    </body>

</html>

In [ ]:
#home.html

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
	<meta charset="UTF-8">
    <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
	<title>Startseite</title>
</head>

<body>
    <h1>Willkommen im Arbeitszeit-Buchungssystem</h1>
    <p><a href="{% url 'login' %}">Zum Login</a></p>
    <p><a href="{% url 'register' %}">zur Registrierung</a></p>
</body>
</html>

In [ ]:
#login.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <meta charset="UTF-8">
        <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    </head>

    <body>
        <h2>Login</h2>

        {% if error %}
        <p style="color:red;">{{ error }}</p>
        {% endif %}

        <form method="POST">
            {% csrf_token %}
            <label>E-Mail:</label>
            <input type="email" name="email" required>

            <label>Passwort:</label>
            <input type="password" name="password" required>

            <button type="submit">Login</button>
        </form>
    </body>
</html>

In [ ]:
#register.html

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <link rel="stylesheet" href="{% static 'meine_app/zeitbuchungssystem.css' %}">
    <title>Registrierung</title>
</head>

<body>
    <h1>Registrierung</h1>
    {% if error %}
        <p styel="color:red;">{{ error }}</p>
    {% endif %}
    <form method="POST">
        {% csrf_token %}
        <label for="username">Benutzername: </label>
        <input type="text" id="username" name="username" required>

        <label for="email">E-Mail: </label>
        <input type="email" id="email" name="email" required>

        <label for="password">Passwort: </label>
        <input type="password" id="password" name="password" required minlength="8">

        <button type="submit">Registrieren</button>
    </form>
</body>

</html>